# 5 — Transmission and trade

Companion to **section 6**. A real twelve-zone network of Denmark, Germany,
Sweden and Norway, and a price *per zone*. Congestion is where section 2's
scarcity rent puts on a geographic costume.

The network ships with the repository as a single file — you load it, you do
not build it. Appendix C of the note documents where it came from.

Before running: this notebook and every later one read the technology-cost
tables, which are built once with `python data/prepare.py --costs-only` from
the note's directory (see the README).

Runtime: three to four minutes — four solves of the twelve-zone network at
336 segments, most of it in the last experiment.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
sys.path.insert(0, str(note / "pipeline"))   # the run scripts' helpers
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

In [ ]:
from model import network

NC = PROCESSED / "network_eur_bz_2024.nc"
n = network.load_network(NC)          # on the note's fuel prices, outages derated, DC links lossy
print("zones:", list(n.buses.index))
print("generators:", len(n.generators), "| lines:", len(n.lines),
      "| links:", len(n.links), "| storage:", len(n.storage_units))
n.generators.groupby("carrier")["p_nom"].sum().div(1e3).round(1).sort_values(ascending=False).to_frame("GW")

## Cutting the year

A year is 8,784 hours and twelve zones; that is a big LP. `network.aggregate`
cuts it to `HOURS` chronological **segments** — stretches of similar hours
carried as one snapshot with their duration as weight — which is how sections
6–9 of the note are solved (1,095 segments there; 336 here).

Every solve of the section charges the 2024 ETS average, 65 EUR/t, so that
the model can be held against the 2024 market.

In [ ]:
HOURS = 336
TAU = 65.0

network.apply_carbon_price(n, TAU)
network.aggregate(n, HOURS)           # segments; see model/network.py for the scheme
network.apply_ramp_limits(n)          # after aggregation: the limits depend on the step
network.solve(n)

# One row per HOUR of the year, each segment repeated for the hours it stands
# for, so every mean and duration curve is weighted by segment length.
prices = network.expand_to_hours(n, n.buses_t.marginal_price)
prices.mean().round(1).sort_values().to_frame("mean price (EUR/MWh)")

Look at the spread. The Nordic hydro zones sit at a flat *water value* — the
shadow price of stored water, section 4's $\theta_t$ at national scale.
Thermal zones swing around it, and Denmark, wired to both, is pulled between
them.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for z in ["DK1", "DK2", "DE", "NO2", "SE3"]:
    ax.plot(range(len(prices)), sorted(prices[z], reverse=True), lw=1.4, label=z)
ax.set_xlabel("hours (sorted)")
ax.set_ylabel("price (EUR/MWh)")
ax.legend(frameon=False)
plt.show()

## Congestion rent

A congested corridor buys cheap and sells dear. Its rent is
$R_\ell=\sum_t w_t\, f_{\ell,t}\,(\lambda_{\text{to},t}-\lambda_{\text{from},t})$,
and it is positive only in hours where the corridor is at its limit — exactly
the scarcity rent of section 2, earned by a wire.

In [ ]:
rents = network.congestion_rent(n)
rents["rent_meur"] = rents["rent_eur"] / 1e6
rents.sort_values("rent_eur", ascending=False)[["rent_meur", "congested_share", "cap_mw"]].round(2).head(10)

## Building a bigger cable

Section 6's expansion experiment: scale the Skagerrak corridor between DK1
and NO2. System cost falls at a decreasing rate; the corridor's rent first
grows with its capacity, then is competed away as the price spread closes.
Three solves, each a fresh network.

In [ ]:
from run_network import corridor_links, EXPANSION_CORRIDOR

a, b = EXPANSION_CORRIDOR
rows = []
for scale in [0.0, 1.0, 2.0]:
    m = network.load_network(NC)
    network.apply_carbon_price(m, TAU)
    network.aggregate(m, HOURS)
    network.apply_ramp_limits(m)
    links = corridor_links(m, a, b)
    m.links.loc[links, "p_nom"] *= scale
    network.solve(m)
    r = network.congestion_rent(m)
    corridor = "–".join(sorted([a, b]))
    hourly = network.expand_to_hours(m, m.buses_t.marginal_price)
    rows.append({
        "scale": scale,
        "system cost (bnEUR)": m.objective / 1e9,
        "corridor rent (MEUR)": r.loc[corridor, "rent_eur"] / 1e6 if corridor in r.index else 0.0,
        "mean |DK1-NO2 spread|": (hourly[b] - hourly[a]).abs().mean(),
    })
pd.DataFrame(rows).set_index("scale").round(2)

The pipeline (`run_network.py`) does two more things before it reads rents
off this network: it scales Norwegian inflow so the solved hydro output
matches the year's actual production, and derates the internal Nordic
corridors to TYNDP's reference transfer capacities. Both are calibration, not
economics, and Appendix C of the note says what they change.

## Your turn

1. Set `TAU = 0` and re-solve. Which zones export more, and what happens to
   the DK1–DE rent? Section 6.3 of the note reads this figure.
2. Hold the model against the market: `spot_2024.csv` has the actual
   day-ahead prices of six zones. Plot the model's DK1 duration curve over
   the real one. Where does the model miss, and which of the section's
   simplifications is to blame?
3. Sum every corridor's rent and compare it with the total generator rent
   (price minus marginal cost, summed). Who earns more from scarcity, the
   wires or the plants?

In [ ]:
# Try it here.